In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from functools import partial
import pandas as pd

In [3]:
from data import COLUMNS
from eval import f1_by_annotator, f1_conditional_selection, f1_majority, jaccard_index, jaccard_sampled
from train_unsloth import get_data, predict, load_model

/home/stefan/Projects/toxic-reasoning/env-unsloth2/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/stefan/Projects/toxic-reasoning/env-unsloth2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 09-29 10:29:50 [__init__.py:216] Automatically detected platform cuda.
WARNING 09-29 10:29:50 [cuda.py:682] Detected different devices in the system: NVIDIA GeForce RTX 3090, NVIDIA GeForce GT 1030. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Initialize

In [5]:
# model = AutoModelForCausalLM.from_pretrained('./outputs/checkpoint-60', device_map='cuda:0')
# tokenizer = AutoTokenizer.from_pretrained(MODEL_KEY)

# model, tokenizer = FastLanguageModel.from_pretrained('./outputs/checkpoint-60', device_map='cuda:0')

# model, tokenizer = load_model('./gptoss/checkpoint-60', attn_implementation="eager")

# MODEL_NAME = 'gemma'
# model, tokenizer = load_model('./gemma/checkpoint-1535')

MODEL_NAME = 'gptoss'
model, tokenizer = load_model('./gptoss/checkpoint-767')

# MODEL_NAME = 'qwen'
# model, tokenizer = load_model('./qwen/checkpoint-769')
# model = model.merge_and_unload()

==((====))==  Unsloth 2025.9.5: Fast Gpt_Oss patching. Transformers: 4.56.1. vLLM: 0.10.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.691 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gpt_Oss does not support SDPA - switching to fast eager.


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.04it/s]


## Evaluation

In [6]:
# TODO: stop doing inference once for each annotator
by_comment_data, by_thread_data, datasets = get_data(tokenizer, max_nr_comments=int(999999), test_only=True)

100%|██████████| 1649/1649 [00:19<00:00, 86.33it/s] 


Split up 9 that were too long otherwise.
Skipped 3 that were still too long after.


Generating test split: 1658 examples [00:00, 4486.38 examples/s]


In [7]:
unique = set((dct['st_id'], dct['question_only']) for dct in datasets['test'])
datasets['test'] = [{'st_id': st_id, 'question_only': qo} for st_id, qo in unique]

In [8]:
print(len(datasets['test']))
print(datasets['test'][0]['question_only'])

550

### Role:
You are an expert on toxic language, specializing in annotating the explicit or implicit toxicity of messages from social media.

### Blank JSON Schema:
{'$defs': {'CommentAnnotation': {'properties': {'message_nr': {'description': 'The number of the message in the thread to which this annotation applies.', 'title': 'Message Nr', 'type': 'integer'}, 'is_toxic': {'$ref': '#/$defs/Toxicity', 'description': 'If this comment might be perceived as toxic.'}, 'is_only_innapropriate': {'description': 'If the comment is only toxic because of an inappropriate or toxic word (rather than what is being said/implied).', 'title': 'Is Only Innapropriate', 'type': 'boolean'}, 'is_counter_speech': {'description': 'If the comment is an argument-based counter of the previous comment, providing an alternative perspective.', 'title': 'Is Counter Speech', 'type': 'boolean'}, 'toxic_reasoning': {'anyOf': [{'$ref': '#/$defs/ToxicReasoning'}, {'type': 'null'}], 'description': 'If toxic and the tox

In [10]:
predictions = predict(model, tokenizer, datasets, by_comment_data)

1it [04:53, 293.87s/it]

{'comment_annotations': [{'message_nr': 1, 'is_toxic': 'Yes/Maybe', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': {'implication': "The group referred to as 'Proud Boys' is inherently evil and manipulative.", 'subject_descr': "'Proud Boys'", 'subject_role': 'another group', 'subject_span': 'they.', 'subject_characteristic': 'Other', 'has_other': False, 'other_descr': '', 'other_role': 'none of the above', 'other_span': 'Lie, lie, lie some more. Personally, I am past them BELIEVEING this shit. I think they are Bigoted, Fascist, Authoritarian Assholes who hide behind rhetoric and bullshit, to hide the fact they are really EVIL. They DGAF. They use force to get their way. They are EVIL for being Republican Extremnists. They are DUMB for attempting to lie their way out of it.', 'category': "the subject's (inherent) qualities, their nature, abilities, etc.", 'impl_span': 'truly EVIL.', 'polarity': 'Negative', 'stereotype': False, 'sarcasm': False, 'when': ['P

2it [05:45, 151.17s/it]

{'comment_annotations': [{'message_nr': 1, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 2, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 3, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 4, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 5, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}]}


3it [06:27, 101.42s/it]

...

{'comment_annotations': [{'message_nr': 1, 'is_toxic': 'Yes/Maybe', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': {'implication': 'The ministers should be punished by death.', 'subject_descr': 'The ministers mentioned in the post.', 'subject_role': 'an individual outside of the conversation', 'subject_span': 'hepsinin', 'subject_characteristic': 'Other', 'has_other': False, 'other_descr': '', 'other_role': 'none of the above', 'other_span': 'Allah belalarını versin hepsinin yargılanacağı günü iple çekiyorum', 'category': "the subject's circumstances, living conditions, physical condition or health, general wellbeing, access to resources, etc.", 'impl_span': 'Allah belalarını versin hepsinin', 'polarity': 'Negative', 'stereotype': False, 'sarcasm': False, 'when': ['Future'], 'author_belief': 0.9, 'author_preference': 0.9, 'author_responsibility': 0.9, 'typical_belief': 0.5, 'typical_preference': 0.5, 'expert_belief': 0.1}}, {'message_nr': 2, 'is_toxic': 

11it [31:00, 150.97s/it]

{'comment_annotations': [{'message_nr': 1, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 2, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 3, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 4, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 5, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 6, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 7, 'is_toxic': 'No', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning': None}, {'message_nr': 8, 'is_toxic': 'Yes/Maybe', 'is_only_innapropriate': False, 'is_counter_speech': False, 'toxic_reasoning':

11it [31:00, 169.13s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.31 GiB. GPU 0 has a total capacity of 23.69 GiB of which 382.69 MiB is free. Including non-PyTorch memory, this process has 23.31 GiB memory in use. Of the allocated memory 22.86 GiB is allocated by PyTorch, and 123.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [8]:
import pickle
pickle.dump(predictions, open(f'./{MODEL_NAME}/predictions.pkl', 'wb'))

In [9]:
# function that retrieves prediction for evaluation
def get_prediction(st_id, st_nr, comment_id, col, optimistic=True):
    pred = predictions[st_id][st_nr]
    if col == 'toxicity':
        return pred['trinary']['_Yes/Maybe']
    if col == 'counternarrative':
        return pred['trinary']['_Counter-speech']

    # if col.endswith('Tokens'):
    #     return [v for k, v in sorted(pred[col].items(), key=lambda p: p[0])]

    if col == 'subjectGroupType':
        return [pred[col][(lambda x: x if x != '_other' else '_Other')(k)] for k in COLUMNS[col].values]
    if col == 'implTemporality':
        return [pred[col][k] for k in COLUMNS[col].values]

    return pred[col]

# f1_maj = f1_majority(data_by_comment, get_prediction)
# f1_con = f1_conditional_selection(data_by_comment, get_prediction, jaccard_index)

In [10]:
test_df = by_comment_data['test']

### Majority evaluation

In [11]:
maj_results = []
for optimism in [True, False]:
    maj_results_df = f1_majority(test_df, partial(get_prediction, optimistic=optimism))
    maj_results.append(maj_results_df)

pd.concat(maj_results).to_csv(f'eval_outputs/{MODEL_NAME}_eval_group1.csv')

### By-annotator evaluation

Here we measure how much the model agrees with each annotator

In [12]:
# results_df = f1_by_annotator(test_df, get_prediction)
# results_df.to_csv(f'{MODEL_NAME}_by_annotator.csv')

### Conditional Evaluation

In [13]:
cond_results = []
for optimism in [True, False]:
    for aggregation in ['none', 'max-score', 'random']:
        results_df = f1_conditional_selection(test_df, partial(get_prediction, optimistic=optimism), jaccard_index, aggregation=aggregation)
        results_df['aggregation'] = aggregation
        results_df['optimistic'] = optimism
        cond_results.append(results_df)

pd.concat(cond_results).to_csv(f'eval_outputs/{MODEL_NAME}_eval_group2b.csv')

subject subjectTokens
cond_scores (8738, 2) 1807
eval_predictions (8738,) 8738
subjectGroupType subjectTokens
cond_scores (8738, 2) 1807
eval_predictions (8738,) 8738
other otherTokens
cond_scores (8738, 2) 388
eval_predictions (8738,) 8738
implTopic implTopicTokens
cond_scores (8738, 2) 1963
eval_predictions (8738,) 8738
implPolarity implTopicTokens
cond_scores (8738, 2) 1963
eval_predictions (8738,) 8738
implTemporality implTopicTokens
cond_scores (8738, 2) 1963
eval_predictions (8738,) 8738
subject subjectTokens
cond_scores (8738, 2) 1807
eval_predictions (8738,) 8738
subjectGroupType subjectTokens
cond_scores (8738, 2) 1807
eval_predictions (8738,) 8738
other otherTokens
cond_scores (8738, 2) 388
eval_predictions (8738,) 8738
implTopic implTopicTokens
cond_scores (8738, 2) 1963
eval_predictions (8738,) 8738
implPolarity implTopicTokens
cond_scores (8738, 2) 1963
eval_predictions (8738,) 8738
implTemporality implTopicTokens
cond_scores (8738, 2) 1963
eval_predictions (8738,) 8738
su

### Jaccard Evaluation

In [14]:
jacc_results = []
for optimism in [True, False]:
    jaccard_df = jaccard_sampled(test_df, partial(get_prediction, optimistic=optimism))
    jaccard_df['optimistic'] = optimism
    jacc_results.append(jaccard_df)
pd.concat(jacc_results).to_csv(f'eval_outputs/{MODEL_NAME}_eval_group2a.csv')

answer_pp_implDetected
False    6711
True     2027
Name: count, dtype: int64


100%|██████████| 50/50 [00:00<00:00, 56.89it/s]


answer_pp_implDetected
False    6711
True     2027
Name: count, dtype: int64


100%|██████████| 50/50 [00:00<00:00, 77.36it/s]
